# Data Science Internship - February 2026
## Task 5: Fine-tune Transformer for POS Tagging and Chunking

This notebook uses the CoNLL-2003 dataset and trains two token-classification models:
- POS Tagging model
- Chunking model


## Task 1: Dataset Selection

**Dataset:** CoNLL-2003 (`conll2003` from Hugging Face Datasets)

**Label categories used:**
- POS labels: from `pos_tags`
- Chunk labels: from `chunk_tags`


In [ ]:
import sys
import subprocess

def pip_install(packages):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *packages])

pip_install([
    'transformers',
    'datasets',
    'evaluate',
    'seqeval',
    'accelerate'
])


In [ ]:
import numpy as np
import evaluate

from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    DataCollatorForTokenClassification,
    Trainer,
    TrainingArguments,
)

MODEL_NAME = 'distilbert-base-uncased'
MAX_LENGTH = 128
LEARNING_RATE = 2e-5
EPOCHS = 2
TRAIN_BATCH_SIZE = 16
EVAL_BATCH_SIZE = 16
SEED = 42


In [ ]:
dataset = load_dataset('conll2003')
print(dataset)

pos_feature = dataset['train'].features['pos_tags'].feature
chunk_feature = dataset['train'].features['chunk_tags'].feature

pos_label_list = pos_feature.names
chunk_label_list = chunk_feature.names

print('Number of POS labels:', len(pos_label_list))
print('Number of Chunk labels:', len(chunk_label_list))
print('POS labels:', pos_label_list)
print('Chunk labels:', chunk_label_list)


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_and_align_labels(examples, label_key):
    tokenized = tokenizer(
        examples['tokens'],
        truncation=True,
        is_split_into_words=True,
        max_length=MAX_LENGTH,
    )

    all_labels = examples[label_key]
    aligned_labels = []

    for i, labels in enumerate(all_labels):
        word_ids = tokenized.word_ids(batch_index=i)
        previous_word_id = None
        label_ids = []

        for word_id in word_ids:
            if word_id is None:
                label_ids.append(-100)  # special tokens
            elif word_id != previous_word_id:
                label_ids.append(labels[word_id])
            else:
                label_ids.append(-100)  # subword continuation
            previous_word_id = word_id

        aligned_labels.append(label_ids)

    tokenized['labels'] = aligned_labels
    return tokenized


In [ ]:
tokenized_pos = dataset.map(
    lambda x: tokenize_and_align_labels(x, 'pos_tags'),
    batched=True,
)

print('POS tokenized columns:', tokenized_pos['train'].column_names)
print('Sample POS labels aligned length:', len(tokenized_pos['train'][0]['labels']))


In [ ]:
tokenized_chunk = dataset.map(
    lambda x: tokenize_and_align_labels(x, 'chunk_tags'),
    batched=True,
)

print('Chunk tokenized columns:', tokenized_chunk['train'].column_names)
print('Sample Chunk labels aligned length:', len(tokenized_chunk['train'][0]['labels']))


In [ ]:
seqeval = evaluate.load('seqeval')

def build_compute_metrics(label_list):
    def compute_metrics(eval_pred):
        logits, labels = eval_pred
        predictions = np.argmax(logits, axis=2)

        true_predictions = []
        true_labels = []

        for pred_row, label_row in zip(predictions, labels):
            filtered_preds = []
            filtered_labels = []
            for p, l in zip(pred_row, label_row):
                if l != -100:
                    filtered_preds.append(label_list[p])
                    filtered_labels.append(label_list[l])
            true_predictions.append(filtered_preds)
            true_labels.append(filtered_labels)

        results = seqeval.compute(predictions=true_predictions, references=true_labels)
        return {
            'precision': results['overall_precision'],
            'recall': results['overall_recall'],
            'f1': results['overall_f1'],
            'accuracy': results['overall_accuracy'],
        }
    return compute_metrics


In [ ]:
id2label_pos = {i: label for i, label in enumerate(pos_label_list)}
label2id_pos = {label: i for i, label in enumerate(pos_label_list)}

model_pos = AutoModelForTokenClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(pos_label_list),
    id2label=id2label_pos,
    label2id=label2id_pos,
)

args_pos = TrainingArguments(
    output_dir='./task5_pos_model',
    eval_strategy='epoch',
    save_strategy='no',
    learning_rate=LEARNING_RATE,
    per_device_train_batch_size=TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=EVAL_BATCH_SIZE,
    num_train_epochs=EPOCHS,
    weight_decay=0.01,
    seed=SEED,
    report_to=[],
)

trainer_pos = Trainer(
    model=model_pos,
    args=args_pos,
    train_dataset=tokenized_pos['train'],
    eval_dataset=tokenized_pos['validation'],
    tokenizer=tokenizer,
    data_collator=DataCollatorForTokenClassification(tokenizer),
    compute_metrics=build_compute_metrics(pos_label_list),
)

# trainer_pos.train()  # Uncomment to train
# pos_eval = trainer_pos.evaluate(tokenized_pos['test'])
# print('POS Test Metrics:', pos_eval)


In [ ]:
id2label_chunk = {i: label for i, label in enumerate(chunk_label_list)}
label2id_chunk = {label: i for i, label in enumerate(chunk_label_list)}

model_chunk = AutoModelForTokenClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(chunk_label_list),
    id2label=id2label_chunk,
    label2id=label2id_chunk,
)

args_chunk = TrainingArguments(
    output_dir='./task5_chunk_model',
    eval_strategy='epoch',
    save_strategy='no',
    learning_rate=LEARNING_RATE,
    per_device_train_batch_size=TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=EVAL_BATCH_SIZE,
    num_train_epochs=EPOCHS,
    weight_decay=0.01,
    seed=SEED,
    report_to=[],
)

trainer_chunk = Trainer(
    model=model_chunk,
    args=args_chunk,
    train_dataset=tokenized_chunk['train'],
    eval_dataset=tokenized_chunk['validation'],
    tokenizer=tokenizer,
    data_collator=DataCollatorForTokenClassification(tokenizer),
    compute_metrics=build_compute_metrics(chunk_label_list),
)

# trainer_chunk.train()  # Uncomment to train
# chunk_eval = trainer_chunk.evaluate(tokenized_chunk['test'])
# print('Chunking Test Metrics:', chunk_eval)


In [ ]:
def predict_tags(sentence_tokens, trainer, label_list):
    inputs = tokenizer(
        sentence_tokens,
        is_split_into_words=True,
        return_tensors='pt',
        truncation=True,
        max_length=MAX_LENGTH,
    )

    outputs = trainer.model(**inputs)
    preds = outputs.logits.argmax(dim=-1).squeeze().tolist()

    word_ids = inputs.word_ids(batch_index=0)
    final_tags = []
    used_word_ids = set()

    for idx, word_id in enumerate(word_ids):
        if word_id is None or word_id in used_word_ids:
            continue
        used_word_ids.add(word_id)
        final_tags.append((sentence_tokens[word_id], label_list[preds[idx]]))

    return final_tags

# Example sentence
sentence = 'John works at Google in California'.split()

# Run these after training (or after loading saved checkpoints)
# print('POS Tags:', predict_tags(sentence, trainer_pos, pos_label_list))
# print('Chunk Tags:', predict_tags(sentence, trainer_chunk, chunk_label_list))


## Task 7 and Task 8: Comparison and Report

**POS Tagging vs Chunking:**
- POS tagging labels each token with a grammatical category (noun, verb, adjective, etc.).
- Chunking groups tokens into shallow phrases (NP, VP, PP), which is a higher-level structural task.

**Why chunking is typically harder:**
- Chunk labels depend on phrase boundaries and BIO format.
- Boundary errors can hurt sequence-level metrics more than simple token-level grammar errors.

**Challenges faced:**
- Label alignment for subword tokenization
- Handling special tokens with `-100`
- Balancing training time vs performance

**Observation:**
POS tagging is generally easier and converges faster; chunking requires stronger sequence understanding.


## LinkedIn Post Draft (Task 5)
**Summary:**
In this NLP task, I fine-tuned a transformer model for token classification to perform POS tagging and chunking. I worked on data preprocessing, label alignment, model training, evaluation, and custom sentence inference.

**Key Learnings:**
- Token classification using transformer models
- Subword tokenization and label alignment with `-100`
- Fine-tuning DistilBERT/BERT using Hugging Face Trainer
- Evaluating sequence labeling with seqeval (precision, recall, F1)
- Difference between grammar-level POS tags and phrase-level chunk tags

**Acknowledgment:**
Thanks to Innomatics Research Labs, my trainer, and my mentor for their guidance and support.

**Hashtags:** #NLP #AI #DataScience #MachineLearning
